In [4]:
import os
import pandas as pd

# ── Change only these two lines to relocate all I/O ──────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"   # all raw input files live here
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/"    # all processed CSVs will be written here
# ─────────────────────────────────────────────────────────────────────────────

# Sub-paths derived automatically – do not edit
EVOLF_IN_PATH  = os.path.join(BASE_PATH, "evolf")
EVOLF_OUT_PATH = os.path.join(OUT_PATH,  "evolf")

# Ensure output directories exist
os.makedirs(EVOLF_OUT_PATH, exist_ok=True)

print("BASE_PATH :", BASE_PATH)
print("OUT_PATH  :", OUT_PATH)


BASE_PATH : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
OUT_PATH  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/


In [ ]:
Pubchem_Syn_fil = pd.read_csv(
    os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered"),
    sep="\t", header=None
)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))
del Pubchem_Syn_fil
print(f"Pubchem_Syn_fil_dict loaded: {len(Pubchem_Syn_fil_dict):,} entries")


In [ ]:
Pubchem = pd.read_pickle(os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl"))
Pubchem_CID_Smile_Dict  = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_SMILES"]))
Pubchem_IUPAC_CID_Dict  = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_IUPAC_NAME"]))
del Pubchem
print(f"Pubchem_CID_Smile_Dict : {len(Pubchem_CID_Smile_Dict):,} entries")
print(f"Pubchem_IUPAC_CID_Dict : {len(Pubchem_IUPAC_CID_Dict):,} entries")


In [ ]:
ENS_2NCBI = pd.read_csv(os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv"))
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI["name"].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI["name"], ENS_2NCBI["initial_alias"]))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info"), sep="\t")
NCBI_ALL_GENE["ENSEMBLE_ID"] = NCBI_ALL_GENE["Symbol"].map(NCBI_2ENS__dict)

NCBI_ALL_GENEname_dict     = dict(zip(NCBI_ALL_GENE["GeneID"],      NCBI_ALL_GENE["description"]))
NCBI_ALL_GENEIDname_dict   = dict(zip(NCBI_ALL_GENE["GeneID"],      NCBI_ALL_GENE["Symbol"]))
NCBI_Synbol_GENEname_dict  = dict(zip(NCBI_ALL_GENE["Symbol"],      NCBI_ALL_GENE["description"]))
NCBI_GENEname__Symbol_dict = dict(zip(NCBI_ALL_GENE["description"], NCBI_ALL_GENE["Symbol"]))
del NCBI_ALL_GENE
print(f"NCBI_Synbol_GENEname_dict  : {len(NCBI_Synbol_GENEname_dict):,} entries")
print(f"NCBI_GENEname__Symbol_dict : {len(NCBI_GENEname__Symbol_dict):,} entries")


EvOlf Ligand / Receptor ID maps

In [ ]:
Ligand_ID_file = pd.read_csv(
    os.path.join(EVOLF_IN_PATH, "Ligands.csv"), encoding="ISO-8859-1"
)
Ligand_ID_file["CIDs"] = (
    Ligand_ID_file["CIDs"].astype(str).str.replace(r"\.0$", "", regex=True)
)
Ligand_ID_file_dict = dict(zip(Ligand_ID_file["Ligand IDs"], Ligand_ID_file["CIDs"]))
del Ligand_ID_file

Receptor_ID_file = pd.read_csv(
    os.path.join(EVOLF_IN_PATH, "Receptors.csv"), encoding="ISO-8859-1"
)
Receptor_ID_file_dict = dict(zip(Receptor_ID_file["Receptor IDs"], Receptor_ID_file["Receptor"]))
del Receptor_ID_file
print(f"Ligand_ID_file_dict   : {len(Ligand_ID_file_dict):,} entries")
print(f"Receptor_ID_file_dict : {len(Receptor_ID_file_dict):,} entries")


# EvOlf Ligand / Receptor ID maps

In [5]:
Ligand_ID_file = pd.read_csv(
    os.path.join(EVOLF_IN_PATH, "Ligands.csv"), encoding="ISO-8859-1"
)
Ligand_ID_file["CIDs"] = (
    Ligand_ID_file["CIDs"].astype(str).str.replace(r"\.0$", "", regex=True)
)
Ligand_ID_file_dict = dict(zip(Ligand_ID_file["Ligand IDs"], Ligand_ID_file["CIDs"]))
del Ligand_ID_file

Receptor_ID_file = pd.read_csv(
    os.path.join(EVOLF_IN_PATH, "Receptors.csv"), encoding="ISO-8859-1"
)
Receptor_ID_file_dict = dict(zip(Receptor_ID_file["Receptor IDs"], Receptor_ID_file["Receptor"]))
del Receptor_ID_file
print(f"Ligand_ID_file_dict   : {len(Ligand_ID_file_dict):,} entries")
print(f"Receptor_ID_file_dict : {len(Receptor_ID_file_dict):,} entries")


Ligand_ID_file_dict   : 38,214 entries
Receptor_ID_file_dict : 2,111 entries


## EvOlf — Olfactory Ligand–Receptor Pairs

In [ ]:
evolf = pd.read_csv(
    os.path.join(EVOLF_IN_PATH, "EvOlf_Pairs.csv"), encoding="ISO-8859-1"
)

evolf.rename(columns={
    "Ligand ID":   "Head",
    "Receptor ID": "Tail",
    "Canonical SMILES": "Head_SEQ",
}, inplace=True)

evolf["Head_ID"] = evolf["Head"].map(Ligand_ID_file_dict)
evolf[["Head", "Head_ID"]] = evolf[["Head_ID", "Head"]]

evolf["Tail_ID"] = evolf["Tail"].map(Receptor_ID_file_dict)
evolf[["Tail", "Tail_ID"]] = evolf[["Tail_ID", "Tail"]]

evolf["Head_detail_name"] = evolf["Head"].map(Pubchem_IUPAC_CID_Dict)
evolf["Head_SMILE_name"]  = evolf["Head"].map(Pubchem_CID_Smile_Dict)
evolf["Tail_detail_name"] = evolf["Tail"].map(NCBI_Synbol_GENEname_dict)

evolf = evolf[evolf["Head_detail_name"].notna()]
evolf = evolf[evolf["Tail_detail_name"].notna()]

evolf["Head_ID_IS"] = np.where(
    evolf["Head"].isna(), None,
    np.where(evolf["Head"].astype(str).str.startswith("DB"), "Drugbank", "Pubchem")
)
# Bug 9 fixed: single assignment (was set to Uniprot_ID then overwritten to NCBI_ID)
evolf["Tail_ID_IS"] = "NCBI_ID"
evolf["Head_type"]  = "ChemicalEntity"
evolf["Tail_type"]  = "Gene"
evolf["Relation"]   = evolf["Head_type"] + "_" + evolf["Tail_type"]
evolf["KG_Source"]  = "Evolf"

desired_cols = [
    "Head", "Relation", "Tail", "Head_type", "Tail_type",
    "KG_Source", "Head_detail_name", "Head_SMILE_name",
    "Tail_detail_name", "Head_ID_IS", "Tail_ID_IS",
]
existing_cols = [c for c in desired_cols if c in evolf.columns]
evolf = evolf[existing_cols]
print(f"Final EvOlf rows: {len(evolf):,}")
evolf

In [ ]:
evolf.to_csv(os.path.join(EVOLF_OUT_PATH, "Chemical_Gene_Evolf.csv"), index=False)
print("EvOlf: saved.")
del evolf
